<a href="https://colab.research.google.com/github/SnehaGummadi/L1TD1_ORF1p_Binding_Sites_Prediction/blob/main/cleaning_training_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Connect to drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import numpy as np

## Fetch Data

In [ ]:
# RIP-seq ORF1p
# BioProject: PRJNA1211128
rip_orf1 = pd.read_csv("/content/drive/MyDrive/deep_learning_in_genomics/orf1p_rip_counts.csv.gz",
                       compression='gzip', header=0, sep="\t", low_memory=False,
                       index_col=0)
rip_orf1.head()

,PDAC8_ORF1p_RIP_1,PDAC8_ORF1p_RIP_2,PDAC8_ORF1p_RIP_3,PDAC8_Input_4,PDAC8_Input_5,PDAC8_Input_6,PDAC6_ORF1p_RIP_7,PDAC6_ORF1p_RIP_8,PDAC6_ORF1p_RIP_9,PDAC6_Input_10,PDAC6_Input_11,PDAC6_Input_12,PDAC3_ORF1p_RIP_13,PDAC3_ORF1p_RIP_14,PDAC3_ORF1p_RIP_15,PDAC3_Input_16,PDAC3_Input_17,PDAC3_Input_18
Name,,,,,,,,,,,,,,,,,,
ENST00000832824.1,0.0,0.0,0.0,0.0,13.047,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832825.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832826.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832827.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832828.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,22.624,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# RIP-seq L1TD1
# BioProject: PRJNA1065915

rip_l1td1 = pd.read_csv("/content/drive/MyDrive/deep_learning_in_genomics/l1td1_rip_counts.csv.gz",
                        compression='gzip', header=0, sep='\t', low_memory=False,
                        index_col=0)
rip_l1td1.head()

,DNMT1_KO_IP_3,DNMT1_KO_IP_2,DNMT1_KO_IP_1,DNMT1_KO_input3,DNMT1_KO_input2,DNMT1_KO_input1
Name,,,,,,
ENST00000832824.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832825.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832826.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832827.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832828.1,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
clip_seq_links = {"L1TD1_IgG_pos" : "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE227855&format=file&file=GSE227855%5FL1TD1%5FIgG%5Fhg19%5Fcoverage%5Fpos%2EbedGraph%2Egz",
                  "L1TD1_IgG_neg" : "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE227855&format=file&file=GSE227855%5FL1TD1%5FIgG%5Fhg19%5Fcoverage%5Fneg%2EbedGraph%2Egz",
                  "L1TD1_hg19_pos" : "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE227855&format=file&file=GSE227855%5FL1TD1%5Fhg19%5Fcoverage%5Fneg%2EbedGraph%2Egz",
                  "L1TD1_hg19_neg" : "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE227855&format=file&file=GSE227855%5FL1TD1%5Fhg19%5Fcoverage%5Fneg%2EbedGraph%2Egz"}

clip_l1td1 = {}

for sample in clip_seq_links.keys():
  df = pd.read_csv(clip_seq_links[sample], comment='#', sep="\t", low_memory=False, compression='gzip', header=None)
  clip_l1td1[sample] = df

clip_l1td1.keys()

## Pre-Process Data

In [ ]:
!pip install pydeseq2 -qq
!pip install pysam -qq
!pip install biopython -qq
#!pip install pybedtools -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.6 MB/s eta 0:00:00


In [ ]:
#!apt-get install -y bedtools

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

### RIP-seq L1TD1

In [ ]:
rip_l1td1.head()

,DNMT1_KO_IP_3,DNMT1_KO_IP_2,DNMT1_KO_IP_1,DNMT1_KO_input3,DNMT1_KO_input2,DNMT1_KO_input1
Name,,,,,,
ENST00000832824.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832825.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832826.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832827.1,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832828.1,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# RIP-seq L1TD1

l1td1_ip = ["DNMT1_KO_IP_3",	"DNMT1_KO_IP_2",	"DNMT1_KO_IP_1"	]
l1td1_input = ["DNMT1_KO_input3", "DNMT1_KO_input2",	"DNMT1_KO_input1"]

print(f"Shape of rip_l1td1: {rip_l1td1.shape}")
rip_l1td1_filtered = rip_l1td1[rip_l1td1.ge(10).sum(axis=1) >= 3]
print(f"Shape of filtered rip_l1td1: {rip_l1td1_filtered.shape}")

rip_l1td1_filtered.head()

Shape of rip_l1td1: (517038, 6)
Shape of filtered rip_l1td1: (24074, 6)


,DNMT1_KO_IP_3,DNMT1_KO_IP_2,DNMT1_KO_IP_1,DNMT1_KO_input3,DNMT1_KO_input2,DNMT1_KO_input1
Name,,,,,,
ENST00000831237.1,0.000,39.811,49.392,13.439,32.502,63.996
ENST00000484859.1,8.536,8.168,12.237,9.975,11.522,10.313
ENST00000506640.4,0.000,0.000,0.000,12.730,10.805,11.242
ENST00000743878.1,13.103,17.189,30.860,0.000,0.000,0.000
ENST00000416931.1,181.050,182.147,98.337,66.546,18.927,43.886


In [ ]:
rip_l1td1_deseq2_counts = rip_l1td1_filtered.T.round().astype(int)
rip_l1td1_deseq2_counts

Name,ENST00000831237.1,ENST00000484859.1,ENST00000506640.4,ENST00000743878.1,ENST00000416931.1,ENST00000457540.1,ENST00000414273.1,ENST00000427426.1,ENST00000514057.1,ENST00000416718.2,...,ENST00000792225.1,ENST00000611339.1,ENST00000849368.1,ENST00000613204.1,ENST00000615165.1,ENST00000818852.1,ENST00000823177.1,ENST00000823337.1,ENST00000826472.1,ENST00000826474.1
DNMT1_KO_IP_3,0,9,0,13,181,21,125,17,2190,32,...,107,11,19,18,45,43,20,9,110,31
DNMT1_KO_IP_2,40,8,0,17,182,24,132,8,2703,42,...,117,35,74,77,70,20,41,0,173,19
DNMT1_KO_IP_1,49,12,0,31,98,26,121,13,2100,32,...,138,27,32,52,101,59,15,16,151,15
DNMT1_KO_input3,13,10,13,0,67,53,122,8,4009,22,...,107,8,3,30,0,0,2,15,131,4
DNMT1_KO_input2,33,12,11,0,19,31,50,9,2483,10,...,187,15,11,79,76,0,3,16,153,4
DNMT1_KO_input1,64,10,11,0,44,52,69,21,3594,11,...,235,21,14,75,61,0,0,29,253,7


In [ ]:
metadata = pd.DataFrame({
    'condition': ['ip', 'ip', 'ip', 'input', 'input', 'input']
}, index=rip_l1td1_deseq2_counts.index)
metadata

,condition
DNMT1_KO_IP_3,ip
DNMT1_KO_IP_2,ip
DNMT1_KO_IP_1,ip
DNMT1_KO_input3,input
DNMT1_KO_input2,input
DNMT1_KO_input1,input


In [ ]:
dds = DeseqDataSet(
    counts=rip_l1td1_deseq2_counts,
    metadata=metadata,
    design_factors='condition',
    ref_level=['condition', 'input']
)
dds.deseq2()

stats = DeseqStats(dds, contrast=['condition', 'ip', 'input'])
stats.summary()
results = stats.results_df

Using None as control genes, passed at DeseqDataSet initialization


/tmp/ipykernel_7378/442242243.py:1: DeprecationWarning: ref_level is deprecated and no longer has any effect. It will beremoved in a future release.
  dds = DeseqDataSet(
/tmp/ipykernel_7378/442242243.py:1: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 41.07 seconds.

Fitting dispersion trend curve...
... done in 1.35 seconds.

Fitting MAP dispersions...
... done in 44.32 seconds.

Fitting LFCs...
... done in 43.43 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 6.38 seconds.



Log2 fold change & Wald test p-value: condition ip vs input
                     baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Name                                                                          
ENST00000831237.1   30.204690       -0.746977  1.487006 -0.502337  0.615431   
ENST00000484859.1   10.094690       -0.475783  0.554176 -0.858541  0.390594   
ENST00000506640.4    6.362168       -6.315080  2.063904 -3.059774  0.002215   
ENST00000743878.1    8.517478        6.426549  2.075221  3.096802  0.001956   
ENST00000416931.1   92.239626        1.594363  0.648158  2.459839  0.013900   
...                       ...             ...       ...       ...       ...   
ENST00000818852.1   18.083926        7.507074  2.155830  3.482219  0.000497   
ENST00000823177.1   11.823376        3.518980  1.060116  3.319430  0.000902   
ENST00000823337.1   14.201476       -1.502028  1.272322 -1.180540  0.237785   
ENST00000826472.1  155.898159       -0.616231  0.260439 -2.366119  0.01

In [ ]:
results.to_csv("/content/drive/MyDrive/deep_learning_in_genomics/rip_l1td1_deseq2_results.csv",
               header=True, index=True, sep='\t')

### RIP-seq ORF1p

In [ ]:
rip_orf1

,PDAC8_ORF1p_RIP_1,PDAC8_ORF1p_RIP_2,PDAC8_ORF1p_RIP_3,PDAC8_Input_4,PDAC8_Input_5,PDAC8_Input_6,PDAC6_ORF1p_RIP_7,PDAC6_ORF1p_RIP_8,PDAC6_ORF1p_RIP_9,PDAC6_Input_10,PDAC6_Input_11,PDAC6_Input_12,PDAC3_ORF1p_RIP_13,PDAC3_ORF1p_RIP_14,PDAC3_ORF1p_RIP_15,PDAC3_Input_16,PDAC3_Input_17,PDAC3_Input_18
Name,,,,,,,,,,,,,,,,,,
ENST00000832824.1,0.0,0.0,0.0,0.0,13.047,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832825.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832826.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832827.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000832828.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,22.624,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENST00000797950.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000819228.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,3.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENST00000819229.1,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# RIP-seq L1TD1
orf1_ip = ["PDAC8_ORF1p_RIP_1",	"PDAC8_ORF1p_RIP_2",	"PDAC8_ORF1p_RIP_3",
           "PDAC6_ORF1p_RIP_7",	"PDAC6_ORF1p_RIP_8", "PDAC6_ORF1p_RIP_9",
           "PDAC3_ORF1p_RIP_13",	"PDAC3_ORF1p_RIP_14",	"PDAC3_ORF1p_RIP_15"]
orf1_input = ["PDAC8_Input_4", "PDAC8_Input_5",	"PDAC8_Input_6",
              "PDAC6_Input_10",	"PDAC6_Input_11",	"PDAC6_Input_12",
              "PDAC3_Input_16", "PDAC3_Input_17",	"PDAC3_Input_18"]

print(f"Shape of rip_l1td1: {rip_orf1.shape}")
rip_orf1_filtered = rip_orf1[rip_orf1.ge(10).sum(axis=1) >= 3]
print(f"Shape of filtered rip_l1td1: {rip_orf1_filtered.shape}")

rip_orf1_deseq2_counts = rip_orf1_filtered.T.round().astype(int)
rip_orf1_deseq2_counts

Shape of rip_l1td1: (517038, 18)
Shape of filtered rip_l1td1: (94396, 18)


Name,ENST00000831279.1,ENST00000831237.1,ENST00000831726.1,ENST00000831718.1,ENST00000466430.5,ENST00000484859.1,ENST00000495576.1,ENST00000442987.3,ENST00000833897.1,ENST00000831142.1,...,ENST00000849367.1,ENST00000613204.1,ENST00000615165.1,ENST00000621424.4,ENST00000817838.1,ENST00000817841.1,ENST00000818004.1,ENST00000826472.1,ENST00000762498.1,ENST00000795710.1
PDAC8_ORF1p_RIP_1,0,13,0,8,12,14,13,0,0,0,...,3781,79,19,0,0,0,2,8,1,0
PDAC8_ORF1p_RIP_2,0,0,3,11,19,11,14,1,0,0,...,3194,68,0,0,11,0,0,0,0,0
PDAC8_ORF1p_RIP_3,0,11,0,14,6,11,18,0,0,0,...,2746,44,0,38,0,0,0,2,0,6
PDAC8_Input_4,0,0,31,0,0,18,0,0,0,0,...,160960,388,79,0,14,11,0,11,1,14
PDAC8_Input_5,0,13,21,0,1,29,26,2,0,46,...,181866,384,0,14,0,0,3,23,1,7
PDAC8_Input_6,0,0,18,4,17,85,12,0,22,0,...,241324,438,0,17,0,0,0,12,0,0
PDAC6_ORF1p_RIP_7,20,0,14,14,3,16,0,2,0,0,...,4018,34,0,0,0,0,0,9,2,0
PDAC6_ORF1p_RIP_8,43,0,2,0,0,4,16,0,0,0,...,3745,0,11,0,0,0,0,3,0,0
PDAC6_ORF1p_RIP_9,27,9,8,0,11,11,11,0,0,107,...,2761,57,4,0,0,0,0,5,0,1
PDAC6_Input_10,0,10,0,9,0,51,1,21,28,0,...,78743,43,0,0,0,0,29,294,21,13


In [ ]:
metadata_orf1 = pd.DataFrame(index=rip_orf1_deseq2_counts.index)
for index, row in metadata_orf1.iterrows():
  if index in orf1_ip:
    metadata_orf1.loc[index, 'condition'] = 'ip'
  else:
    metadata_orf1.loc[index, 'condition'] = 'input'
metadata_orf1

,condition
PDAC8_ORF1p_RIP_1,ip
PDAC8_ORF1p_RIP_2,ip
PDAC8_ORF1p_RIP_3,ip
PDAC8_Input_4,input
PDAC8_Input_5,input
PDAC8_Input_6,input
PDAC6_ORF1p_RIP_7,ip
PDAC6_ORF1p_RIP_8,ip
PDAC6_ORF1p_RIP_9,ip
PDAC6_Input_10,input


In [ ]:
dds_orf1 = DeseqDataSet(
    counts=rip_orf1_deseq2_counts,
    metadata=metadata_orf1,
    design_factors='condition',
    ref_level=['condition', 'input']
)
dds_orf1.deseq2()

stats_orf1 = DeseqStats(dds_orf1, contrast=['condition', 'ip', 'input'])
stats_orf1.summary()
results_orf1 = stats_orf1.results_df

/tmp/ipykernel_7378/3953339545.py:1: DeprecationWarning: ref_level is deprecated and no longer has any effect. It will beremoved in a future release.
  dds_orf1 = DeseqDataSet(
/tmp/ipykernel_7378/3953339545.py:1: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds_orf1 = DeseqDataSet(
Fitting size factors...
... done in 0.13 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 171.58 seconds.

Fitting dispersion trend curve...
... done in 4.41 seconds.

Fitting MAP dispersions...
... done in 184.67 seconds.

Fitting LFCs...
... done in 214.43 seconds.

Calculating cook's distance...
... done in 0.19 seconds.

Replacing 9677 outlier genes.

Fitting dispersions...
... done in 11.89 seconds.

Fitting MAP dispersions...
... done in 7.86 seconds.

Fitting LFCs...
... done in 22.29 seconds.

Running Wald tests...
... done in 28.78 seconds.



Log2 fold change & Wald test p-value: condition ip vs input
                     baseMean  log2FoldChange     lfcSE      stat  \
Name                                                                
ENST00000831279.1   12.317349       -1.179587  2.796824 -0.421759   
ENST00000831237.1    3.923787       -0.685049  1.963866 -0.348826   
ENST00000831726.1    7.128289       -2.212962  1.065905 -2.076134   
ENST00000831718.1    3.779754       -0.181862  1.976763 -0.092000   
ENST00000466430.5    7.538035       -0.048239  0.781062 -0.061761   
...                       ...             ...       ...       ...   
ENST00000817841.1    0.507342       -3.033642  3.145288 -0.964504   
ENST00000818004.1   11.401804       -6.469734  1.885649 -3.431038   
ENST00000826472.1  105.103406       -6.408525  0.913222 -7.017489   
ENST00000762498.1   11.351762       -6.457130  1.411393 -4.575004   
ENST00000795710.1    5.704013       -4.390166  1.597446 -2.748240   

                         pvalue          p

In [ ]:
results_orf1.to_csv("/content/drive/MyDrive/deep_learning_in_genomics/rip_orf1_deseq2_results.csv",
               header=True, index=True, sep='\t')

## Get RNA (DNA) sequences of ORF1p and L1TD1 bound transcripts

In [ ]:
!pip install biopython -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.5 MB/s eta 0:00:00


In [ ]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
import gzip

In [ ]:
results_l1td1 = pd.read_csv("/content/drive/MyDrive/deep_learning_in_genomics/rip_l1td1_deseq2_results.csv",
               header=0, sep='\t')
results_l1td1.head()

,Name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,ENST00000831237.1,30.204690,-0.746977,1.487006,-0.502337,0.615431,0.791604
1,ENST00000484859.1,10.094690,-0.475783,0.554176,-0.858541,0.390594,0.606252
2,ENST00000506640.4,6.362168,-6.315080,2.063904,-3.059774,0.002215,0.008867
3,ENST00000743878.1,8.517478,6.426549,2.075221,3.096802,0.001956,0.007952
4,ENST00000416931.1,92.239626,1.594363,0.648158,2.459839,0.013900,0.043788


In [ ]:
results_orf1 = pd.read_csv("/content/drive/MyDrive/deep_learning_in_genomics/rip_orf1_deseq2_results.csv",
               header=0, sep='\t')
results_orf1.head()

,Name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
0,ENST00000831279.1,12.317349,-1.179587,2.796824,-0.421759,0.673201,0.918857
1,ENST00000831237.1,3.923787,-0.685049,1.963866,-0.348826,0.727220,0.936900
2,ENST00000831726.1,7.128289,-2.212962,1.065905,-2.076134,0.037882,0.187624
3,ENST00000831718.1,3.779754,-0.181862,1.976763,-0.092000,0.926698,0.986067
4,ENST00000466430.5,7.538035,-0.048239,0.781062,-0.061761,0.950753,0.991602


In [ ]:
bound_l1td1 = results_l1td1[
    (results_l1td1['log2FoldChange'] > 1) &
     (results_l1td1['padj'] < 0.05)
]

unbound_l1td1 = results_l1td1[
    (results_l1td1['log2FoldChange'] < -1) &
     (results_l1td1['padj'] < 0.05)
]

print(f"bound shape: {bound_l1td1.shape}")
print(f"unbound shape: {unbound_l1td1.shape}")

bound shape: (4274, 7)
unbound shape: (3040, 7)


In [ ]:
# Get list of genes for bound and unbound
bound_l1td1_genes = bound_l1td1['Name'].tolist()
unbound_l1td1_genes = unbound_l1td1['Name'].tolist()

In [ ]:
bound_orf1 = results_orf1[
    (results_orf1['log2FoldChange'] > 1) &
     (results_orf1['padj'] < 0.05)
]

unbound_orf1 = results_orf1[
    (results_orf1['log2FoldChange'] < -1) &
     (results_orf1['padj'] < 0.05)
]

print(f"bound shape: {bound_orf1.shape}")
print(f"unbound shape: {unbound_orf1.shape}")

bound shape: (2587, 7)
unbound shape: (6270, 7)


In [ ]:
# Get list of genes for bound and unbound
bound_orf1_genes = bound_orf1['Name'].tolist()
unbound_orf1_genes = unbound_orf1['Name'].tolist()

In [ ]:
# %%bash
# wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.transcripts.fa.gz

In [ ]:
# Read transcriptome file and get the nucleotide sequences

with gzip.open("/content/drive/MyDrive/deep_learning_in_genomics/gencode.v49.transcripts.fa.gz", "rt") as handle:
  transcriptome_dict = SeqIO.to_dict(SeqIO.parse(handle, "fasta"))


In [ ]:
transcript_names = list(transcriptome_dict.keys())
transcript_names[-10:-1]

['ENST00000797955.1|ENSG00000303902.1|-|-|ENST00000797955|ENSG00000303902|1607|lncRNA|',
 'ENST00000797957.1|ENSG00000303902.1|-|-|ENST00000797957|ENSG00000303902|938|lncRNA|',
 'ENST00000797956.1|ENSG00000303902.1|-|-|ENST00000797956|ENSG00000303902|920|lncRNA|',
 'ENST00000797948.1|ENSG00000303902.1|-|-|ENST00000797948|ENSG00000303902|985|lncRNA|',
 'ENST00000797950.1|ENSG00000303902.1|-|-|ENST00000797950|ENSG00000303902|1315|lncRNA|',
 'ENST00000797959.1|ENSG00000303902.1|-|-|ENST00000797959|ENSG00000303902|1378|lncRNA|',
 'ENST00000819228.1|ENSG00000306528.1|-|-|ENST00000819228|ENSG00000306528|567|lncRNA|',
 'ENST00000819229.1|ENSG00000306528.1|-|-|ENST00000819229|ENSG00000306528|391|lncRNA|',
 'ENST00000751368.1|ENSG00000297844.1|-|-|ENST00000751368|ENSG00000297844|769|lncRNA|']

In [ ]:
# Extract keys from transcript_names that correspond with the bound genes

bound_l1td1_transcript_names = [name for name in transcript_names if name.split('|')[0] in bound_l1td1_genes]
unbound_l1td1_transcript_names = [name for name in transcript_names if name.split('|')[0] in unbound_l1td1_genes]
bound_orf1_transcript_names = [name for name in transcript_names if name.split('|')[0] in bound_orf1_genes]
unbound_orf1_transcript_names = [name for name in transcript_names if name.split('|')[0] in unbound_orf1_genes]

In [ ]:
# Create a dataframe for l1td1 and orf1
sequences = [str(transcriptome_dict[name].seq) for name in bound_l1td1_transcript_names] + \
            [str(transcriptome_dict[name].seq) for name in unbound_l1td1_transcript_names]
labels = [1] * len(bound_l1td1_transcript_names) + \
         [0] * len(unbound_l1td1_transcript_names)

l1td1_seq_df = pd.DataFrame({
    'transcript_sequence': sequences,
    'bound': labels
})
l1td1_seq_df.head()

,transcript_sequence,bound
0,ACCGTGAGGGAGGAACAGGATCGCACTCGGGCTGCTGGGAGGCCCC...,1
1,TTTGACCTTCAGCAAGGTCAAAGGGAGTCCGAACTAGTCTCAGGCT...,1
2,GCTGCTCCCGAGTCGGCGCGCGGCGGGGACGCGAGTCCGTAGTTTC...,1
3,GGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCCG...,1
4,ACTCGCGAGTCCGGCCTGGGCCGCCGGCCCGGCGCGGGCGCCATGA...,1


In [ ]:
# Create a dataframe for l1td1 and orf1
sequences = [str(transcriptome_dict[name].seq) for name in bound_orf1_transcript_names] + \
            [str(transcriptome_dict[name].seq) for name in unbound_orf1_transcript_names]
labels = [1] * len(bound_orf1_transcript_names) + \
         [0] * len(unbound_orf1_transcript_names)

orf1_seq_df = pd.DataFrame({
    'transcript_sequence': sequences,
    'bound': labels
})
orf1_seq_df.head()

,transcript_sequence,bound
0,GGGGCCGGAAGTGGGGTGCACGCTTCGGGTTGGTGTCATGGCAGCT...,1
1,GGCATTCTGGGGCCGGAAGTGGGGTGCACGCTTCGGGTTGGTGTCA...,1
2,AGGGGCCCGCCGCGCCCATCCCGATGGCTGGAGGCGTCTGAGGGGC...,1
3,CTGCAGGCGGGGCGGTGTCGGCGGCCGGAGCCCCCGCGCGGGCCGC...,1
4,AGGCGGGGCGGTGTCGGCGGCCGGAGCCCCCGCGCGGGCCGCCTAT...,1


In [ ]:
l1td1_seq_df.to_csv("/content/drive/MyDrive/deep_learning_in_genomics/l1td1_seq_df.csv",
               header=True, index=False, sep='\t')
orf1_seq_df.to_csv("/content/drive/MyDrive/deep_learning_in_genomics/orf1_seq_df.csv",
               header=True, index=False, sep='\t')

In [ ]:
records = (
    SeqRecord(
        Seq(transcriptome_dict[name].seq),
        id=name,
        description=transcriptome_dict[name].description
    )
    for name in bound_l1td1_transcript_names
  )

with gzip.open("/content/drive/MyDrive/deep_learning_in_genomics/bound_l1td1_transcripts.fa.gz", "wt") as output_handle:
  SeqIO.write(records, output_handle, "fasta")

records = (
    SeqRecord(
        Seq(transcriptome_dict[name].seq),
        id=name,
        description=transcriptome_dict[name].description
    )
    for name in unbound_l1td1_transcript_names
)

with gzip.open("/content/drive/MyDrive/deep_learning_in_genomics/unbound_l1td1_transcripts.fa.gz", "wt") as output_handle:
  SeqIO.write(records, output_handle, "fasta")

records = (
    SeqRecord(
        Seq(transcriptome_dict[name].seq),
        id=name,
        description=transcriptome_dict[name].description
    )
    for name in bound_orf1_transcript_names
)

with gzip.open("/content/drive/MyDrive/deep_learning_in_genomics/bound_orf1_transcripts.fa.gz", "wt") as output_handle:
  SeqIO.write(records, output_handle, "fasta")


records = (
    SeqRecord(
        Seq(transcriptome_dict[name].seq),
        id=name,
        description=transcriptome_dict[name].description
    )
    for name in unbound_orf1_transcript_names
)

with gzip.open("/content/drive/MyDrive/deep_learning_in_genomics/unbound_orf1_transcripts.fa.gz", "wt") as output_handle:
  SeqIO.write(records, output_handle, "fasta")

In [ ]:
# %$bash
# cd /content/drive/MyDrive/deep_learning_in_genomics
# wget https://ftp.ensembl.org/pub/release-115/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
# gunzip Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz

In [ ]:
# Index the fasta file for bedtools using pysam
# import pysam

# pysam.faidx("/content/drive/MyDrive/deep_learning_in_genomics/Homo_sapiens.GRCh38.dna.primary_assembly.fa")

## Model Dependencies

In [ ]:
%pip install wandb -Uq
%pip install nbformat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os

from sklearn.metrics import (roc_auc_score, f1_score,
                              precision_score, recall_score,
                              accuracy_score, confusion_matrix)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import wandb

if torch.backends.mps.is_available():
    torch.set_default_dtype(torch.float32)
    print("Set default to float32 for MPS compatibility")

In [ ]:
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    elif torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Random seed set as {seed}")

set_seed(42)

Random seed set as 42


In [ ]:
DEVICE = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available()
                      else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# Login to W&B
wandb.login()

True

## Load Data

In [ ]:
l1td1_seq_df = pd.read_csv("/content/drive/MyDrive/deep_learning_in_genomics/l1td1_seq_df.csv",
               header=0, sep='\t')
l1td1_seq_df.head()

,transcript_sequence,bound
0,ACCGTGAGGGAGGAACAGGATCGCACTCGGGCTGCTGGGAGGCCCC...,1
1,TTTGACCTTCAGCAAGGTCAAAGGGAGTCCGAACTAGTCTCAGGCT...,1
2,GCTGCTCCCGAGTCGGCGCGCGGCGGGGACGCGAGTCCGTAGTTTC...,1
3,GGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCCG...,1
4,ACTCGCGAGTCCGGCCTGGGCCGCCGGCCCGGCGCGGGCGCCATGA...,1


In [ ]:
l1td1_seq_df.shape

(7314, 2)

In [ ]:
l1td1_seq_df['bound'].value_counts()

,count
bound,
1,4274
0,3040


In [ ]:
# Look at the sequence length distribution
seq_lengths = l1td1_seq_df['transcript_sequence'].apply(len)
seq_lengths.describe().round(2)

,transcript_sequence
count,7314.00
mean,2661.51
std,5472.40
min,69.00
25%,877.00
50%,1614.00
75%,3196.75
max,347561.00


In [ ]:
total = 7314
df = l1td1_seq_df.copy()
over_10k = (df['transcript_sequence'].str.len() > 10000).sum()
print(f"Sequences > 10kb: {over_10k} ({over_10k/total*100:.1f}%)")

# Check if removal is biased toward bound or unbound
df['length'] = df['transcript_sequence'].str.len()
print(df[df['length'] > 10000]['bound'].value_counts())


lengths = df['transcript_sequence'].str.len()
print()
print(lengths[lengths > 50000].describe())
print()
print(df[lengths > 50000]['bound'].value_counts())

Sequences > 10kb: 200 (2.7%)
bound
1    197
0      3
Name: count, dtype: int64

count         3.000000
mean     214746.666667
std      128224.442328
min       91667.000000
25%      148339.500000
50%      205012.000000
75%      276286.500000
max      347561.000000
Name: transcript_sequence, dtype: float64

bound
1    3
Name: count, dtype: int64


In [ ]:
# Check what characters are in the sequences
unique_chars = set()
for seq in df['transcript_sequence']:
  unique_chars.update(seq)
print(f"Unique chars in seqs: {unique_chars}")

Unique chars in seqs: {'A', 'N', 'T', 'C', 'G'}


In [ ]:
# Drop transcripts that have a length greater than 21,000 bps
lengths = df['transcript_sequence'].str.len()
print()
print(lengths[lengths > 21000].describe())
print()
print(df[lengths > 21000]['bound'].value_counts())


count         9.000000
mean      88303.444444
std      114549.924772
min       21103.000000
25%       21777.000000
50%       28227.000000
75%       91667.000000
max      347561.000000
Name: transcript_sequence, dtype: float64

bound
1    9
Name: count, dtype: int64


In [ ]:
df.groupby('bound')['length'].describe()

,count,mean,std,min,25%,50%,75%,max
bound,,,,,,,,
0,3040.0,1369.504605,1025.440567,116.0,707.00,1110.0,1706.00,13327.0
1,4274.0,3580.476603,6962.236351,69.0,1202.25,2455.0,4715.25,347561.0


In [ ]:
# Drop transcripts that have a length greater than 21,000 bps
print(l1td1_seq_df.shape)
l1td1_seq_df = l1td1_seq_df[l1td1_seq_df['transcript_sequence'].str.len() <= 14000]
print(l1td1_seq_df.shape)

(7269, 2)
(7269, 2)


## Encode Data

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

def dynamic_pad_collate(batch):
    """Pads a batch of variable-length pre-computed tensors."""
    sequences = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    max_len = max([seq.shape[0] for seq in sequences])

    padded_sequences = []
    for seq in sequences:
        padded = torch.zeros((max_len, 4), dtype=torch.float32)
        padded[:seq.shape[0], :] = seq
        padded_sequences.append(padded)

    return torch.stack(padded_sequences), torch.stack(labels)

In [ ]:
def one_hot_encode(seq):
    """One-hot encode a sequence to its natural length without global padding."""
    allowed = set("ACTGN U")
    if not set(seq).issubset(allowed):
        invalid = set(seq) - allowed
        raise ValueError(f"Invalid characters in sequence: {invalid}")

    nuc_d = {'A': [1.0, 0.0, 0.0, 0.0],
             'C': [0.0, 1.0, 0.0, 0.0],
             'G': [0.0, 0.0, 1.0, 0.0],
             'T': [0.0, 0.0, 0.0, 1.0],
             'N': [0.0, 0.0, 0.0, 0.0]}

    seq_len = len(seq)
    encoded = np.zeros((seq_len, 4), dtype=np.float32)

    for i, base in enumerate(seq):
        encoded[i] = nuc_d[base]

    return encoded

In [ ]:
class SeqDatasetPreloaded(Dataset):
    """
    High-RAM Dataset: Pre-computes and stores all tensors in memory
    during initialization for maximum training speed.
    """
    def __init__(self, df, seq_col='transcript_sequence', label_col='bound'):
        self.samples = []
        self.labels = []

        print(f"Pre-computing one-hot encoding for {len(df)} sequences...")

        # Iterate with a progress bar
        for _, row in tqdm(df.iterrows(), total=len(df)):
            seq = row[seq_col]
            label = row[label_col]

            # Encode to a numpy array, then convert to a PyTorch tensor
            # We store natural lengths, e.g., Tensor of shape (1250, 4)
            encoded_tensor = torch.tensor(one_hot_encode(seq))
            label_tensor = torch.tensor(label, dtype=torch.float32)

            self.samples.append(encoded_tensor)
            self.labels.append(label_tensor)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Instant access; no string processing required!
        return self.samples[idx], self.labels[idx]

In [ ]:
def build_dataloaders_preloaded(train_df, val_df, test_df, batch_size=64):

    print("--- Loading Training Set ---")
    train_ds = SeqDatasetPreloaded(train_df)

    print("--- Loading Validation Set ---")
    val_ds   = SeqDatasetPreloaded(val_df)

    print("--- Loading Test Set ---")
    test_ds  = SeqDatasetPreloaded(test_df)

    n_bound = train_df['bound'].sum()
    n_unbound = len(train_df) - n_bound
    print(f"\nTraining set -> Bound: {n_bound}, Unbound: {n_unbound}, Ratio: {n_bound/n_unbound:.2f}")

    class_weights = {0: 1.0 / n_unbound, 1: 1.0 / n_bound}
    sample_weights = [class_weights[label] for label in train_df['bound'].values]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # Increase num_workers if your CPU has multiple cores.
    # Because there is no string encoding happening, workers just move memory quickly.
    train_dl = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=dynamic_pad_collate,
        drop_last=True,
        num_workers=4,
        pin_memory=True # Speeds up transfer from RAM to GPU
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=batch_size,
        collate_fn=dynamic_pad_collate,
        num_workers=4,
        pin_memory=True
    )

    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        collate_fn=dynamic_pad_collate,
        num_workers=4,
        pin_memory=True
    )

    return train_dl, val_dl, test_dl

In [ ]:
train_df, test_df = train_test_split(l1td1_seq_df, test_size=0.2, random_state=42)
train_df, val_df  = train_test_split(train_df,  test_size=0.2, random_state=42)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

Train: (4652, 2)
Val:   (1163, 2)
Test:  (1454, 2)


## Model

In [ ]:
class RBP_Binary_CNN(nn.Module):
    def __init__(self, num_filters=32, num_filters2=32, num_filters3=32, kernel_size=10):
        super().__init__()
        self.conv    = nn.Conv1d(4, num_filters, kernel_size=kernel_size)
        self.norm    = nn.BatchNorm1d(num_filters)
        self.relu    = nn.GELU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(num_filters, num_filters2, kernel_size=kernel_size)
        self.norm2 = nn.BatchNorm1d(num_filters2)
        self.relu2 = nn.GELU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.1)

        self.conv3 = nn.Conv1d(num_filters2, num_filters3, kernel_size=kernel_size)
        self.norm3 = nn.BatchNorm1d(num_filters3)
        self.relu3 = nn.GELU()
        self.drop = nn.Dropout1d(0.5)

        self.globalpool = nn.AdaptiveMaxPool1d(1)
        self.linear  = nn.Linear(num_filters3, 1)

    def forward(self, xb):
        xb = xb.permute(0, 2, 1)   # (batch, seq_len, 4) → (batch, 4, seq_len)
        x = self.pool1(self.relu(self.norm(self.conv(xb))))
        x = self.pool2(self.relu2(self.norm2(self.conv2(x))))
        x = self.drop2(x)
        x = self.relu3(self.norm3(self.conv3(x)))
        x = self.drop(x)
        x = self.globalpool(x).squeeze(-1)
        out = self.linear(x)

        return out

In [ ]:
class EarlyStopping:
    """
    Stops training when val_loss stops improving and saves the best weights.

    Args:
        patience:  how many epochs to wait after last improvement
        min_delta: minimum change to count as an improvement
        path:      where to save the best model checkpoint
    """
    def __init__(self, patience=10, min_delta=1e-4, path='best_model.pt'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.path       = path

        self.best_loss  = np.inf
        self.counter    = 0        # epochs without improvement
        self.best_epoch = 0
        self.stop       = False    # flag the training loop reads

    def step(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.min_delta:
            # Improvement — save weights and reset counter
            self.best_loss  = val_loss
            self.best_epoch = epoch
            self.counter    = 0
            torch.save(model.state_dict(), self.path)
            print(f"  ✓ New best val_loss: {val_loss:.4f} — checkpoint saved")
        else:
            # No improvement
            self.counter += 1
            print(f"  ✗ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.stop = True

    def load_best_weights(self, model):
        """Call after training to restore the best checkpoint."""
        model.load_state_dict(torch.load(self.path))
        print(f"  Restored best weights from epoch {self.best_epoch}")

In [ ]:
def train_model(model, train_dl, val_dl, device, lr=0.01, epochs=50,
                optimizer_cls=torch.optim.SGD,
                scheduler_patience=5, scheduler_factor=0.5, scheduler_min_lr=1e-6,
                # --- early stopping params ---
                es_patience=10, es_min_delta=1e-4, checkpoint_path='best_model.pt'):

    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=1e-4)
    if optimizer_cls == torch.optim.SGD:
      optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, nesterov=True)

    loss_fn   = nn.BCEWithLogitsLoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=scheduler_factor,
        patience=scheduler_patience, min_lr=scheduler_min_lr
    )

    early_stopping = EarlyStopping(
        patience=es_patience,
        min_delta=es_min_delta,
        path=checkpoint_path
    )

    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        batch_losses, batch_sizes = [], []
        all_train_preds, all_train_labels = [], []

        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb.float()).squeeze(-1)
            loss   = loss_fn(logits, yb.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())
            batch_sizes.append(len(xb))
            preds = (torch.sigmoid(logits) >= 0.5).long()
            all_train_preds.append(preds.cpu())
            all_train_labels.append(yb.long().cpu())

        train_loss = np.average(batch_losses, weights=batch_sizes)
        train_losses.append(train_loss)
        train_acc  = (torch.cat(all_train_preds).numpy() ==
                      torch.cat(all_train_labels).numpy()).mean()

        # --- Validation ---
        model.eval()
        vl, ns = [], []
        all_val_preds, all_val_labels = [], []

        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb.float()).squeeze(-1)
                loss   = loss_fn(logits, yb.float())
                vl.append(loss.item())
                ns.append(len(xb))
                preds = (torch.sigmoid(logits) >= 0.5).long()
                all_val_preds.append(preds.cpu())
                all_val_labels.append(yb.long().cpu())

        val_loss = np.average(vl, weights=ns)
        val_losses.append(val_loss)
        val_acc  = (torch.cat(all_val_preds).numpy() ==
                    torch.cat(all_val_labels).numpy()).mean()

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"E{epoch+1:03d} | "
              f"train loss: {train_loss:.4f}  acc: {train_acc:.4f} | "
              f"val loss: {val_loss:.4f}  acc: {val_acc:.4f} | "
              f"lr: {current_lr:.2e}")

        wandb.log({
            'epoch':      epoch + 1,
            'train_loss': train_loss,
            'train_acc':  train_acc,
            'val_loss':   val_loss,
            'val_acc':    val_acc,
            'lr':         current_lr,
        }, step=epoch+1)

        # --- Early stopping check (after scheduler, at end of epoch) ---
        early_stopping.step(val_loss, model, epoch + 1)
        if early_stopping.stop:
            print(f"\nEarly stopping at epoch {epoch + 1}. "
                  f"Best val_loss: {early_stopping.best_loss:.4f} "
                  f"at epoch {early_stopping.best_epoch}")
            break

    # Restore the best weights before returning
    early_stopping.load_best_weights(model)
    wandb.log({'best_epoch': early_stopping.best_epoch,
               'best_val_loss': early_stopping.best_loss}, step=epoch+1)

    return train_losses, val_losses

In [ ]:
def test_model(model, test_dl, device):
    """Evaluate binary classifier on test set and log metrics to W&B."""
    model.eval()
    all_logits, all_labels = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device)
            logits = model(xb.float()).squeeze(-1)   # raw logits, shape (batch,)
            all_logits.append(logits.cpu())
            all_labels.append(yb)

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probs = 1 / (1 + np.exp(-logits))       # sigmoid → probabilities
    preds = (probs >= 0.5).astype(int)       # hard predictions

    metrics = {
        'test_accuracy':  accuracy_score(labels, preds),
        'test_auroc':     roc_auc_score(labels, probs),      # uses probabilities, not hard preds
        'test_f1':        f1_score(labels, preds),
        'test_precision': precision_score(labels, preds),
        'test_recall':    recall_score(labels, preds),
    }

    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    print("\n=== Test Results ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print(f"  Confusion matrix → TN:{tn}  FP:{fp}  FN:{fn}  TP:{tp}")

    wandb.log(metrics)
    return probs, preds, labels, metrics

In [ ]:
# train_dl, val_dl, test_dl = build_dataloaders(train_df, val_df, test_df)

# print("Dataloads Built...")

# model = RBP_Binary_CNN(num_filters=16,
#                         num_filters2=8,
#                         kernel_size=10
#                         ).to(DEVICE)

# print("Training started...")

# train_model(model, train_dl, val_dl, DEVICE, lr=0.01,
#             epochs=50, optimizer_cls=torch.optim.SGD
#             )

# test_model(model, test_dl, DEVICE)

In [ ]:
!pip install graphviz -qq
!pip install torchview -qq

## Hyperparameter Tuning

In [ ]:
train_dl, val_dl, test_dl = build_dataloaders_preloaded(train_df, val_df, test_df)

--- Loading Training Set ---
Pre-computing one-hot encoding for 4652 sequences...


100%|██████████| 4652/4652 [00:18<00:00, 245.28it/s]


--- Loading Validation Set ---
Pre-computing one-hot encoding for 1163 sequences...


100%|██████████| 1163/1163 [00:04<00:00, 259.87it/s]


--- Loading Test Set ---
Pre-computing one-hot encoding for 1454 sequences...


100%|██████████| 1454/1454 [00:03<00:00, 404.52it/s]


Training set -> Bound: 2692, Unbound: 1960, Ratio: 1.37



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'test_auroc', 'goal':'maximize'},
    'parameters':{
        'num_filters': {'values': [32, 64, 128, 256]},
        'num_filters2': {'values': [32]},
        'kernel_size': {'values': [6, 15, 20]},
        'learning_rate': {'values': [0.001, 0.0005]},
        'optimizer': {'values': ['SGD']}
    }
}

In [ ]:
sweep_id = wandb.sweep(sweep_config, project='final_project')
print(f"Sweep ID: {sweep_id}")

Create sweep with ID: 4qvpcs7j
Sweep URL: https://wandb.ai/sneha-gumball-university-of-chicago/final_project/sweeps/4qvpcs7j
Sweep ID: 4qvpcs7j


In [ ]:
def train_sweep():
  with wandb.init(project='final_project'):
    config = wandb.config

    model = RBP_Binary_CNN(num_filters=config.num_filters,
                           num_filters2=config.num_filters2,
                           kernel_size=config.kernel_size
                           ).to(DEVICE)

    optimizer_cls = torch.optim.SGD if config.optimizer == 'SGD' else torch.optim.Adam

    train_model(model, train_dl, val_dl, DEVICE, lr=config.learning_rate,
                epochs=50, optimizer_cls=optimizer_cls
                )

    test_model(model, test_dl, DEVICE)


In [ ]:
wandb.agent(sweep_id, train_sweep, count=3)

wandb: Agent Starting Run: rxnmt24s with config:
wandb: 	kernel_size: 15
wandb: 	learning_rate: 0.001
wandb: 	num_filters: 32
wandb: 	num_filters2: 32
wandb: 	optimizer: SGD
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


E001 | train loss: 0.8015  acc: 0.5082 | val loss: 0.6914  acc: 0.6191 | lr: 1.00e-03
  ✓ New best val_loss: 0.6914 — checkpoint saved
E002 | train loss: 0.6933  acc: 0.5260 | val loss: 0.6725  acc: 0.6767 | lr: 1.00e-03
  ✓ New best val_loss: 0.6725 — checkpoint saved
E003 | train loss: 0.6817  acc: 0.5551 | val loss: 0.6431  acc: 0.7352 | lr: 1.00e-03
  ✓ New best val_loss: 0.6431 — checkpoint saved
E004 | train loss: 0.6517  acc: 0.6144 | val loss: 0.5747  acc: 0.7180 | lr: 1.00e-03
  ✓ New best val_loss: 0.5747 — checkpoint saved
E005 | train loss: 0.6308  acc: 0.6484 | val loss: 0.5567  acc: 0.7326 | lr: 1.00e-03
  ✓ New best val_loss: 0.5567 — checkpoint saved
E006 | train loss: 0.6114  acc: 0.6786 | val loss: 0.5592  acc: 0.7309 | lr: 1.00e-03
  ✗ No improvement (1/10)
E007 | train loss: 0.6022  acc: 0.6888 | val loss: 0.5500  acc: 0.7291 | lr: 1.00e-03
  ✓ New best val_loss: 0.5500 — checkpoint saved
E008 | train loss: 0.5993  acc: 0.6992 | val loss: 0.5584  acc: 0.7240 | lr: 1

best_epoch,▁
best_val_loss,▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,████████████████▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▁▁▁▁
test_accuracy,▁
test_auroc,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▁▂▄▅▅▆▆▆▆▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇████▇███
+3,...


wandb: Agent Starting Run: dvfgubwz with config:
wandb: 	kernel_size: 6
wandb: 	learning_rate: 0.0005
wandb: 	num_filters: 256
wandb: 	num_filters2: 32
wandb: 	optimizer: SGD
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

best_epoch,▁
best_val_loss,▁
epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
lr,█████████████▃▃▃▃▃▃▃▃▃▃▃▁▁▁▁▁
test_accuracy,▁
test_auroc,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▂▂▃▄▆▆▆▇▆▇▇▇▇▇▇▇▇█▇█▇██▇████
+3,...


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 0mzqx173 with config:
wandb: 	kernel_size: 6
wandb: 	learning_rate: 0.0005
wandb: 	num_filters: 32
wandb: 	num_filters2: 32
wandb: 	optimizer: SGD
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


E001 | train loss: 0.8596  acc: 0.5171 | val loss: 0.6823  acc: 0.6612 | lr: 5.00e-04
  ✓ New best val_loss: 0.6823 — checkpoint saved
E002 | train loss: 0.6868  acc: 0.5371 | val loss: 0.6703  acc: 0.6991 | lr: 5.00e-04
  ✓ New best val_loss: 0.6703 — checkpoint saved
E003 | train loss: 0.6768  acc: 0.5727 | val loss: 0.6797  acc: 0.5555 | lr: 5.00e-04
  ✗ No improvement (1/10)
E004 | train loss: 0.6711  acc: 0.5911 | val loss: 0.6542  acc: 0.6948 | lr: 5.00e-04
  ✓ New best val_loss: 0.6542 — checkpoint saved
E005 | train loss: 0.6602  acc: 0.6081 | val loss: 0.6374  acc: 0.7300 | lr: 5.00e-04
  ✓ New best val_loss: 0.6374 — checkpoint saved
E006 | train loss: 0.6527  acc: 0.6204 | val loss: 0.6021  acc: 0.6750 | lr: 5.00e-04
  ✓ New best val_loss: 0.6021 — checkpoint saved
E007 | train loss: 0.6507  acc: 0.6246 | val loss: 0.6073  acc: 0.7352 | lr: 5.00e-04
  ✗ No improvement (1/10)
E008 | train loss: 0.6285  acc: 0.6482 | val loss: 0.5941  acc: 0.7395 | lr: 5.00e-04
  ✓ New best va

best_epoch,▁
best_val_loss,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,██████████████████████████████▃▃▃▃▃▃▃▃▃▁
test_accuracy,▁
test_auroc,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▂▃▃▄▄▅▆▆▆▆▆▆▆▇▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇▇▇█▇███▇██
+3,...
